In [7]:
import pandas as pd
import numpy as np

In [8]:
data = pd.read_csv("../../server/survey_export_pilot_2.csv")

In [9]:
survey_metrics = ['mental_demand', 'success_level', 'frustration_level', 
                  'trajectory_choice_ease', 'difference_clarity', 'preference_learning']
    
# First filter by expected choice counts per test_type
if 'tabletop' in data['test_type'].unique():
    tabletop_mask = (data['test_type'] == 'tabletop')
    data.loc[tabletop_mask, 'choices'] = data.loc[tabletop_mask, 'choices'].apply(
        lambda x: x if isinstance(x, str) and len(eval(x)) == 5 else np.nan
    )
if 'robot_nav' in data['test_type'].unique():
    robot_nav_mask = (data['test_type'] == 'robot_nav')
    data.loc[robot_nav_mask, 'choices'] = data.loc[robot_nav_mask, 'choices'].apply(
        lambda x: x if isinstance(x, str) and len(eval(x)) == 6 else np.nan
    )
data = data.dropna(subset=['choices'])

# Then filter for users who completed all conditions (0,1,2) for each test_type
# and have all survey metrics filled
valid_users = []
for user_id, user_data in data.groupby('user_id'):
    # Check if user has all conditions (0,1,2) for each test_type they attempted
    test_types = user_data['test_type'].unique()
    valid = True
    for test in test_types:
        if len(user_data[user_data['test_type'] == test]['condition_number'].unique()) < 3:
            valid = False
            break
    
    # Check all survey metrics are present and non-null
    if valid and not user_data[survey_metrics].isnull().any().any():
        valid_users.append(user_id)
        
print(f"Number of valid users: {len(valid_users)}")

data = data[data['user_id'].isin(valid_users)]

Number of valid users: 4


In [11]:
data[(data['test_type'] == 'robot_nav') & (data['condition_number'] == 2)]

,user_id,participant_id,test_type,condition_number,choices,response_times,mental_demand,success_level,frustration_level,trajectory_choice_ease,difference_clarity,preference_learning,decision_factors,created_at
4,2,1941,robot_nav,2,"[0, 0, 1, 1, 0, 1]","[4.338, 15.994, 3.292, 8.887, 2.586, 12.763]",5.0,3.0,1.0,3.0,3.0,3.0,Safety and efficiency.,2025-09-02 22:47:52.508963
10,3,2183,robot_nav,2,"[0, 0, 1, 1, 1, 1]","[3.285, 4.631, 6.708, 6.538, 7.04, 9.448]",6.0,6.0,2.0,6.0,6.0,4.0,speed,2025-09-02 22:52:06.559310
16,1,5864,robot_nav,2,"[0, 0, 1, 1, 1, 1]","[2.499, 1.879, 1.68, 2.082, 2.026, 1.543]",2.0,4.0,1.0,2.0,1.0,1.0,time taken and distance,2025-09-02 22:37:10.341149
22,5,7501,robot_nav,2,"[1, 1, 1, 1, 1, 1]","[24.349, 19.392, 4.579, 5.223, 98.055, 5.452]",6.0,1.0,6.0,1.0,1.0,1.0,"Avoid Grass and Asphalt, Travel as fast as you...",2025-09-03 03:43:14.478661
